<a href="https://colab.research.google.com/github/Gaurav1205-CSE/Network-Intrusion-Detection-/blob/main/PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import sys
import sklearn
import io
import random

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NSL_KDD_Train.csv to NSL_KDD_Train.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NSL_KDD_Test.csv to NSL_KDD_Test.csv


In [ ]:
train_url = pd.read_csv('NSL_KDD_Train.csv')
test_url = pd.read_csv('NSL_KDD_Test.csv')

print(train_url.head())
print(test_url.head())

   0  tcp ftp_data   SF  491   0.1  0.2  0.3  0.4  0.5  ...   25  0.17  0.03  \
0  0  udp    other   SF  146     0    0    0    0    0  ...    1  0.00  0.60   
1  0  tcp  private   S0    0     0    0    0    0    0  ...   26  0.10  0.05   
2  0  tcp     http   SF  232  8153    0    0    0    0  ...  255  1.00  0.00   
3  0  tcp     http   SF  199   420    0    0    0    0  ...  255  1.00  0.00   
4  0  tcp  private  REJ    0     0    0    0    0    0  ...   19  0.07  0.07   

   0.17.1  0.25  0.26  0.27  0.05  0.28   normal  
0    0.88  0.00  0.00  0.00   0.0  0.00   normal  
1    0.00  0.00  1.00  1.00   0.0  0.00  neptune  
2    0.03  0.04  0.03  0.01   0.0  0.01   normal  
3    0.00  0.00  0.00  0.00   0.0  0.00   normal  
4    0.00  0.00  0.00  0.00   1.0  1.00  neptune  

[5 rows x 42 columns]
   0   tcp   private   REJ    0.1    0.2  0.3  0.4  0.5  0.6  ...  10.1  \
0  0   tcp   private   REJ      0      0    0    0    0    0  ...     1   
1  2   tcp  ftp_data    SF  12983      0

In [ ]:
col_names = ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"]


df = pd.read_csv('NSL_KDD_Train.csv', header=None, names=col_names)
df_test = pd.read_csv('NSL_KDD_Test.csv', header=None, names=col_names)


print('Dimensions of the Training set:',df.shape)
print('Dimensions of the Test set:',df_test.shape)

Dimensions of the Training set: (125973, 42)
Dimensions of the Test set: (22544, 42)


In [ ]:
df.head(5)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune
3,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal


In [ ]:
print('Label distribution Training set:')
print(df['label'].value_counts())
print()
print('Label distribution Test set:')
print(df_test['label'].value_counts())

Label distribution Training set:
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Label distribution Test set:
label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattac

In [ ]:
print('Training set:')
for col_name in df.columns:
    if df[col_name].dtypes == 'object' :
        unique_cat = len(df[col_name].unique())
        print("Feature '{col_name}' has {unique_cat} categories".format(col_name=col_name, unique_cat=unique_cat))

print()
print('Distribution of categories in service:')
print(df['service'].value_counts().sort_values(ascending=False).head())

Training set:
Feature 'protocol_type' has 3 categories
Feature 'service' has 70 categories
Feature 'flag' has 11 categories
Feature 'label' has 23 categories

Distribution of categories in service:
service
http        40338
private     21853
domain_u     9043
smtp         7313
ftp_data     6860
Name: count, dtype: int64


In [ ]:
print('Test set:')
for col_name in df_test.columns:
    if df_test[col_name].dtypes == 'object' :
        unique_cat = len(df_test[col_name].unique())
        print("Feature '{col_name}' has {unique_cat} categories".format(col_name=col_name, unique_cat=unique_cat))

Test set:
Feature 'protocol_type' has 3 categories
Feature 'service' has 64 categories
Feature 'flag' has 11 categories
Feature 'label' has 38 categories


In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
categorical_columns=['protocol_type', 'service', 'flag']

df_categorical_values = df[categorical_columns]
testdf_categorical_values = df_test[categorical_columns]

df_categorical_values.head()

,protocol_type,service,flag
0,tcp,ftp_data,SF
1,udp,other,SF
2,tcp,private,S0
3,tcp,http,SF
4,tcp,http,SF


In [ ]:
# protocol type
unique_protocol=sorted(df.protocol_type.unique())
string1 = 'Protocol_type_'
unique_protocol2=[string1 + x for x in unique_protocol]
print(unique_protocol2)

# service
unique_service=sorted(df.service.unique())
string2 = 'service_'
unique_service2=[string2 + x for x in unique_service]
print(unique_service2)


# flag
unique_flag=sorted(df.flag.unique())
string3 = 'flag_'
unique_flag2=[string3 + x for x in unique_flag]
print(unique_flag2)


dumcols=unique_protocol2 + unique_service2 + unique_flag2


unique_service_test=sorted(df_test.service.unique())
unique_service2_test=[string2 + x for x in unique_service_test]
testdumcols=unique_protocol2 + unique_service2_test + unique_flag2

['Protocol_type_icmp', 'Protocol_type_tcp', 'Protocol_type_udp']
['service_IRC', 'service_X11', 'service_Z39_50', 'service_aol', 'service_auth', 'service_bgp', 'service_courier', 'service_csnet_ns', 'service_ctf', 'service_daytime', 'service_discard', 'service_domain', 'service_domain_u', 'service_echo', 'service_eco_i', 'service_ecr_i', 'service_efs', 'service_exec', 'service_finger', 'service_ftp', 'service_ftp_data', 'service_gopher', 'service_harvest', 'service_hostnames', 'service_http', 'service_http_2784', 'service_http_443', 'service_http_8001', 'service_imap4', 'service_iso_tsap', 'service_klogin', 'service_kshell', 'service_ldap', 'service_link', 'service_login', 'service_mtp', 'service_name', 'service_netbios_dgm', 'service_netbios_ns', 'service_netbios_ssn', 'service_netstat', 'service_nnsp', 'service_nntp', 'service_ntp_u', 'service_other', 'service_pm_dump', 'service_pop_2', 'service_pop_3', 'service_printer', 'service_private', 'service_red_i', 'service_remote_job', 'ser

In [ ]:
df_categorical_values_enc=df_categorical_values.apply(LabelEncoder().fit_transform)

print(df_categorical_values.head())
print('--------------------')
print(df_categorical_values_enc.head())

testdf_categorical_values_enc=testdf_categorical_values.apply(LabelEncoder().fit_transform)

  protocol_type   service flag
0           tcp  ftp_data   SF
1           udp     other   SF
2           tcp   private   S0
3           tcp      http   SF
4           tcp      http   SF
--------------------
   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9


In [ ]:
enc = OneHotEncoder(categories='auto')
df_categorical_values_encenc = enc.fit_transform(df_categorical_values_enc)
df_cat_data = pd.DataFrame(df_categorical_values_encenc.toarray(),columns=dumcols)


# test set
testdf_categorical_values_encenc = enc.fit_transform(testdf_categorical_values_enc)
testdf_cat_data = pd.DataFrame(testdf_categorical_values_encenc.toarray(),columns=testdumcols)

df_cat_data.head()

,Protocol_type_icmp,Protocol_type_tcp,Protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


ADDING MISSING COlUMNS

In [ ]:
trainservice=df['service'].tolist()
testservice= df_test['service'].tolist()
difference=list(set(trainservice) - set(testservice))
string = 'service_'
difference=[string + x for x in difference]
difference

['service_aol',
 'service_red_i',
 'service_http_8001',
 'service_http_2784',
 'service_harvest',
 'service_urh_i']

In [ ]:
for col in difference:
    testdf_cat_data[col] = 0

print(df_cat_data.shape)
print(testdf_cat_data.shape)

(125973, 84)
(22544, 84)


In [ ]:
newdf=df.join(df_cat_data)
newdf.drop('flag', axis=1, inplace=True)
newdf.drop('protocol_type', axis=1, inplace=True)
newdf.drop('service', axis=1, inplace=True)

# test data
newdf_test=df_test.join(testdf_cat_data)
newdf_test.drop('flag', axis=1, inplace=True)
newdf_test.drop('protocol_type', axis=1, inplace=True)
newdf_test.drop('service', axis=1, inplace=True)

print(newdf.shape)
print(newdf_test.shape)

(125973, 123)
(22544, 123)


The dataset was split into separate datasets for each attack category. Attack labels were renamed for each: 0=Normal, 1=DoS, 2=Probe, 3=R2L, 4=U2R

In [ ]:
labeldf=newdf['label']
labeldf_test=newdf_test['label']


# change the label column
newlabeldf=labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})
newlabeldf_test=labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})



# put the new label column back
newdf['label'] = newlabeldf
newdf_test['label'] = newlabeldf_test


/tmp/ipykernel_478/1231765836.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf=labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
/tmp/ipykernel_478/1231765836.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf_test=labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpsto

### Re-initializing and processing `newdf` and `newdf_test`

To ensure `newdf` and `newdf_test` are correctly defined and processed, I am re-running the necessary steps from earlier cells. This includes joining categorical data, dropping original categorical columns, and remapping the 'label' column to numerical attack categories. Afterwards, I will execute the label mapping to 5 categories and binary 'normal'/'attack' labels.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 1. Define Column Names (from OnC6LzIhiNFx)
col_names = ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"]

# 2. Load Dataset (from OnC6LzIhiNFx)
df = pd.read_csv('NSL_KDD_Train.csv', header=None, names=col_names)
df_test = pd.read_csv('NSL_KDD_Test.csv', header=None, names=col_names)

# Print dimensions for verification
print('Dimensions of the Training set:',df.shape)
print('Dimensions of the Test set:',df_test.shape)

# 3. Identify Categorical Columns (from EBTbfpFpkvuS)
categorical_columns=['protocol_type', 'service', 'flag']

df_categorical_values = df[categorical_columns]
testdf_categorical_values = df_test[categorical_columns]

# 4. Apply Label Encoding to categorical features (from Y7ORtbl2lLHx)
df_categorical_values_enc = df_categorical_values.apply(LabelEncoder().fit_transform)
testdf_categorical_values_enc = testdf_categorical_values.apply(LabelEncoder().fit_transform)

# 5. Apply One-Hot Encoding and ensure consistent columns (modified from tG3DZRbSntOp, _0Qc3RmGoCNJ, 5H__qUuFoDHR)
enc = OneHotEncoder(categories='auto', sparse_output=False, handle_unknown='ignore') # handle_unknown='ignore' to prevent errors for unseen categories in test

# Fit on training data and transform both train and test
df_categorical_values_ohe = enc.fit_transform(df_categorical_values_enc)
# Get feature names after OHE for consistency
ohe_feature_names = enc.get_feature_names_out(categorical_columns)

df_cat_data = pd.DataFrame(df_categorical_values_ohe, columns=ohe_feature_names)

# Transform test data using the encoder fitted on training data
testdf_categorical_values_ohe = enc.transform(testdf_categorical_values_enc)
testdf_cat_data = pd.DataFrame(testdf_categorical_values_ohe, columns=ohe_feature_names)

# The following part is the original content of cell 2726995b and fd2c86cf, combined.
# Re-initialize newdf and newdf_test by joining categorical data
newdf = df.join(df_cat_data)
newdf.drop('flag', axis=1, inplace=True)
newdf.drop('protocol_type', axis=1, inplace=True)
newdf.drop('service', axis=1, inplace=True)

# test data
newdf_test = df_test.join(testdf_cat_data)
newdf_test.drop('flag', axis=1, inplace=True)
newdf_test.drop('protocol_type', axis=1, inplace=True)
newdf_test.drop('service', axis=1, inplace=True)

print('Dimensions of newdf:', newdf.shape)
print('Dimensions of newdf_test:', newdf_test.shape)

# Re-apply label mapping from cell zV4LI5dso0tt
labeldf = newdf['label']
labeldf_test = newdf_test['label']

newlabeldf = labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})
newlabeldf_test = labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})

newdf['label'] = newlabeldf
newdf_test['label'] = newlabeldf_test


# Code from cell fd2c86cf to be executed after newdf and newdf_test are defined
# Dictionary to map numerical labels (0-4) back to descriptive string labels
label_to_category_map = {
    0: 'normal',
    1: 'DoS',
    2: 'Probe',
    3: 'R2L',
    4: 'U2R'
}

# Create 'labels5' column (5-category string labels) from the existing numerical 'label' column
newdf['labels5'] = newdf['label'].map(label_to_category_map)
newdf_test['labels5'] = newdf_test['label'].map(label_to_category_map)

# Create 'labels2' column for binary classification (normal vs attack)
newdf['labels2'] = newdf['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
newdf_test['labels2'] = newdf_test['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')


print('Distribution of labels2 in Training set:')
print(newdf['labels2'].value_counts())
print('\nDistribution of labels2 in Test set:')
print(newdf_test['labels2'].value_counts())

print('\nDistribution of labels5 in Training set:')
print(newdf['labels5'].value_counts())
print('\nDistribution of labels5 in Test set:')
print(newdf_test['labels5'].value_counts())

Dimensions of the Training set: (125973, 42)
Dimensions of the Test set: (22544, 42)
Dimensions of newdf: (125973, 123)
Dimensions of newdf_test: (22544, 123)
Distribution of labels2 in Training set:
labels2
normal    67343
attack    58630
Name: count, dtype: int64

Distribution of labels2 in Test set:
labels2
attack    12833
normal     9711
Name: count, dtype: int64

Distribution of labels5 in Training set:
labels5
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Distribution of labels5 in Test set:
labels5
normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64


/tmp/ipykernel_25719/3306939511.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf = labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
/tmp/ipykernel_25719/3306939511.py:72: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf_test = labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1

### Attack Category Mapping and Indexing (Pandas)

First, I'll define the `attack_dict` to map specific attack types to broader categories (normal, DoS, Probe, R2L, U2R). Then, I'll create `labels2` (normal/attack) and `labels5` (5 categories) columns, followed by Label Encoding these new categorical labels into numerical indices.

In [ ]:
# Dictionary to map numerical labels (0-4) back to descriptive string labels
# Based on the mapping in cell zV4LI5dso0tt:
# 'normal' : 0
# 'neptune', 'back', 'land', 'pod', 'smurf', 'teardrop', 'mailbomb', 'apache2', 'processtable', 'udpstorm', 'worm'  => DoS (1)
# 'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint' => Probe (2)
# 'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy', 'warezclient', 'warezmaster', 'sendmail', 'named', 'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop', 'httptunnel' => R2L (3)
# 'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'ps', 'sqlattack', 'xterm' => U2R (4)

label_to_category_map = {
    0: 'normal',
    1: 'DoS',
    2: 'Probe',
    3: 'R2L',
    4: 'U2R'
}

# Create 'labels5' column (5-category string labels) from the existing numerical 'label' column
newdf['labels5'] = newdf['label'].map(label_to_category_map)
newdf_test['labels5'] = newdf_test['label'].map(label_to_category_map)

# Create 'labels2' column for binary classification (normal vs attack)
newdf['labels2'] = newdf['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
newdf_test['labels2'] = newdf_test['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')


print('Distribution of labels2 in Training set:')
print(newdf['labels2'].value_counts())
print('\nDistribution of labels2 in Test set:')
print(newdf_test['labels2'].value_counts())

print('\nDistribution of labels5 in Training set:')
print(newdf['labels5'].value_counts())
print('\nDistribution of labels5 in Test set:')
print(newdf_test['labels5'].value_counts())

Distribution of labels2 in Training set:
labels2
normal    67343
attack    58630
Name: count, dtype: int64

Distribution of labels2 in Test set:
labels2
attack    12833
normal     9711
Name: count, dtype: int64

Distribution of labels5 in Training set:
labels5
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Distribution of labels5 in Test set:
labels5
normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd # Ensure pandas is imported as it's used for concat

# Initialize LabelEncoders
le_labels2 = LabelEncoder()
le_labels5 = LabelEncoder()

# Fit and transform 'labels2'
# Combine train and test labels to ensure consistent encoding
all_labels2 = pd.concat([newdf['labels2'], newdf_test['labels2']]).unique()
le_labels2.fit(all_labels2)
pickle.dump(le_labels2, open('le_labels2.pkl', 'wb')) # Save le_labels2

newdf['labels2_index'] = le_labels2.transform(newdf['labels2'])
newdf_test['labels2_index'] = le_labels2.transform(newdf_test['labels2'])

# Fit and transform 'labels5'
# Combine train and test labels to ensure consistent encoding
all_labels5 = pd.concat([newdf['labels5'], newdf_test['labels5']]).unique()
le_labels5.fit(all_labels5)

newdf['labels5_index'] = le_labels5.transform(newdf['labels5'])
newdf_test['labels5_index'] = le_labels5.transform(newdf_test['labels5'])

print('Labels2 Index Mapping:')
for i, label in enumerate(le_labels2.classes_):
    print(f'{label}: {i}')

print('\nLabels5 Index Mapping:')
for i, label in enumerate(le_labels5.classes_):
    print(f'{label}: {i}')

print('\nFirst 5 rows of newdf with new label columns:')
display(newdf[['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']].head())

print('\nFirst 5 rows of newdf_test with new label columns:')
display(newdf_test[['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']].head())

Labels2 Index Mapping:
attack: 0
normal: 1

Labels5 Index Mapping:
DoS: 0
Probe: 1
R2L: 2
U2R: 3
normal: 4

First 5 rows of newdf with new label columns:


,label,labels2,labels2_index,labels5,labels5_index
0,0,normal,1,normal,4
1,0,normal,1,normal,4
2,1,attack,0,DoS,0
3,0,normal,1,normal,4
4,0,normal,1,normal,4



First 5 rows of newdf_test with new label columns:


,label,labels2,labels2_index,labels5,labels5_index
0,1,attack,0,DoS,0
1,1,attack,0,DoS,0
2,0,normal,1,normal,4
3,2,attack,0,Probe,1
4,2,attack,0,Probe,1


In [ ]:
to_drop_DoS = [0,1]
to_drop_Probe = [0,2]
to_drop_R2L = [0,3]
to_drop_U2R = [0,4]


DoS_df=newdf[newdf['label'].isin(to_drop_DoS)];
Probe_df=newdf[newdf['label'].isin(to_drop_Probe)];
R2L_df=newdf[newdf['label'].isin(to_drop_R2L)];
U2R_df=newdf[newdf['label'].isin(to_drop_U2R)];



#test
DoS_df_test=newdf_test[newdf_test['label'].isin(to_drop_DoS)];
Probe_df_test=newdf_test[newdf_test['label'].isin(to_drop_Probe)];
R2L_df_test=newdf_test[newdf_test['label'].isin(to_drop_R2L)];
U2R_df_test=newdf_test[newdf_test['label'].isin(to_drop_U2R)];


print('Train:')
print('Dimensions of DoS:' ,DoS_df.shape)
print('Dimensions of Probe:' ,Probe_df.shape)
print('Dimensions of R2L:' ,R2L_df.shape)
print('Dimensions of U2R:' ,U2R_df.shape)
print()
print('Test:')
print('Dimensions of DoS:' ,DoS_df_test.shape)
print('Dimensions of Probe:' ,Probe_df_test.shape)
print('Dimensions of R2L:' ,R2L_df_test.shape)
print('Dimensions of U2R:' ,U2R_df_test.shape)

Train:
Dimensions of DoS: (113270, 127)
Dimensions of Probe: (78999, 127)
Dimensions of R2L: (68338, 127)
Dimensions of U2R: (67395, 127)

Test:
Dimensions of DoS: (17171, 127)
Dimensions of Probe: (12132, 127)
Dimensions of R2L: (12596, 127)
Dimensions of U2R: (9778, 127)


In [ ]:
label_cols_to_drop = ['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']

X_DoS = DoS_df.drop(columns=label_cols_to_drop)
Y_DoS = DoS_df['label']

X_Probe = Probe_df.drop(columns=label_cols_to_drop)
Y_Probe = Probe_df['label']

X_R2L = R2L_df.drop(columns=label_cols_to_drop)
Y_R2L = R2L_df['label']

X_U2R = U2R_df.drop(columns=label_cols_to_drop)
Y_U2R = U2R_df['label']

# Test set
X_DoS_test = DoS_df_test.drop(columns=label_cols_to_drop)
Y_DoS_test = DoS_df_test['label']

X_Probe_test = Probe_df_test.drop(columns=label_cols_to_drop)
Y_Probe_test = Probe_df_test['label']

X_R2L_test = R2L_df_test.drop(columns=label_cols_to_drop)
Y_R2L_test = R2L_df_test['label']

X_U2R_test = U2R_df_test.drop(columns=label_cols_to_drop)
Y_U2R_test = U2R_df_test['label']

In [ ]:
colNames=list(X_DoS)
colNames_test=list(X_DoS_test)

In [ ]:
from sklearn import preprocessing

scaler1 = preprocessing.StandardScaler().fit(X_DoS)
X_DoS=scaler1.transform(X_DoS)

scaler2 = preprocessing.StandardScaler().fit(X_Probe)
X_Probe=scaler2.transform(X_Probe)

scaler3 = preprocessing.StandardScaler().fit(X_R2L)
X_R2L=scaler3.transform(X_R2L)

scaler4 = preprocessing.StandardScaler().fit(X_U2R)
X_U2R=scaler4.transform(X_U2R)

# test data
scaler5 = preprocessing.StandardScaler().fit(X_DoS_test)
X_DoS_test=scaler5.transform(X_DoS_test)

scaler6 = preprocessing.StandardScaler().fit(X_Probe_test)
X_Probe_test=scaler6.transform(X_Probe_test)

scaler7 = preprocessing.StandardScaler().fit(X_R2L_test)
X_R2L_test=scaler7.transform(X_R2L_test)

scaler8 = preprocessing.StandardScaler().fit(X_U2R_test)
X_U2R_test=scaler8.transform(X_U2R_test)

NameError: name 'X_DoS' is not defined

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier


clf = RandomForestClassifier(n_estimators=10,n_jobs=2)
rfe = RFE(estimator=clf, n_features_to_select=13, step=1)

rfe.fit(X_DoS, Y_DoS.astype(int))
X_rfeDoS=rfe.transform(X_DoS)
true=rfe.support_
rfecolindex_DoS=[i for i, x in enumerate(true) if x]
rfecolname_DoS=list(colNames[i] for i in rfecolindex_DoS)

In [ ]:
rfe.fit(X_Probe, Y_Probe.astype(int))
X_rfeProbe=rfe.transform(X_Probe)
true=rfe.support_
rfecolindex_Probe=[i for i, x in enumerate(true) if x]
rfecolname_Probe=list(colNames[i] for i in rfecolindex_Probe)

In [ ]:
rfe.fit(X_R2L, Y_R2L.astype(int))
X_rfeR2L=rfe.transform(X_R2L)
true=rfe.support_
rfecolindex_R2L=[i for i, x in enumerate(true) if x]
rfecolname_R2L=list(colNames[i] for i in rfecolindex_R2L)

In [ ]:
rfe.fit(X_U2R, Y_U2R.astype(int))
X_rfeU2R=rfe.transform(X_U2R)
true=rfe.support_
rfecolindex_U2R=[i for i, x in enumerate(true) if x]
rfecolname_U2R=list(colNames[i] for i in rfecolindex_U2R)

In [ ]:
print('Features selected for DoS:',rfecolname_DoS)
print()
print('Features selected for Probe:',rfecolname_Probe)
print()
print('Features selected for R2L:',rfecolname_R2L)
print()
print('Features selected for U2R:',rfecolname_U2R)

Features selected for DoS: ['src_bytes', 'dst_bytes', 'wrong_fragment', 'count', 'srv_count', 'same_srv_rate', 'diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'Protocol_type_icmp', 'service_ecr_i', 'flag_SF']

Features selected for Probe: ['src_bytes', 'dst_bytes', 'count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'Protocol_type_tcp', 'service_eco_i', 'service_private']

Features selected for R2L: ['duration', 'src_bytes', 'dst_bytes', 'hot', 'is_guest_login', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_rerror_rate', 'service_ftp_data']

Features selected for U2R: ['duration', 'src_bytes', 'dst_bytes', 'hot', 'num_compromised', 'root_shell', 'num_file_creations', 'count', 

In [ ]:
print(X_rfeDoS.shape)
print(X_rfeProbe.shape)
print(X_rfeR2L.shape)
print(X_rfeU2R.shape)

(113270, 13)
(78999, 13)
(68338, 13)
(67395, 13)


In [ ]:
# all features
clf_DoS=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_Probe=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_R2L=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_U2R=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_DoS.fit(X_DoS, Y_DoS.astype(int))
clf_Probe.fit(X_Probe, Y_Probe.astype(int))
clf_R2L.fit(X_R2L, Y_R2L.astype(int))
clf_U2R.fit(X_U2R, Y_U2R.astype(int))

RandomForestClassifier(n_estimators=10, n_jobs=2)

In [ ]:
# selected features
clf_rfeDoS=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_rfeProbe=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_rfeR2L=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_rfeU2R=RandomForestClassifier(n_estimators=10,n_jobs=2)
clf_rfeDoS.fit(X_rfeDoS, Y_DoS.astype(int))
clf_rfeProbe.fit(X_rfeProbe, Y_Probe.astype(int))
clf_rfeR2L.fit(X_rfeR2L, Y_R2L.astype(int))
clf_rfeU2R.fit(X_rfeU2R, Y_U2R.astype(int))

RandomForestClassifier(n_estimators=10, n_jobs=2)

In [ ]:
clf_DoS.predict(X_DoS_test)

array([1, 1, 0, ..., 0, 0, 0])

In [ ]:
clf_DoS.predict_proba(X_DoS_test)[0:10]

array([[0.4, 0.6],
       [0.4, 0.6],
       [1. , 0. ],
       [1. , 0. ],
       [0.7, 0.3],
       [1. , 0. ],
       [0.8, 0.2],
       [0.4, 0.6],
       [0.5, 0.5],
       [1. , 0. ]])

In [ ]:
Y_DoS_pred=clf_DoS.predict(X_DoS_test)
pd.crosstab(Y_DoS_test, Y_DoS_pred, rownames=['Actual attacks'], colnames=['Predicted attacks'])

Predicted attacks,0,1
Actual attacks,,
0,9691,20
1,2897,4563


In [ ]:
Y_Probe_pred=clf_Probe.predict(X_Probe_test)
pd.crosstab(Y_Probe_test, Y_Probe_pred, rownames=['Actual attacks'], colnames=['Predicted attacks'])

Predicted attacks,0,2
Actual attacks,,
0,8974,737
2,1146,1275


In [ ]:
Y_R2L_pred=clf_R2L.predict(X_R2L_test)
pd.crosstab(Y_R2L_test, Y_R2L_pred, rownames=['Actual attacks'], colnames=['Predicted attacks'])

Predicted attacks,0,3
Actual attacks,,
0,9711,0
3,2884,1


In [ ]:
Y_U2R_pred=clf_U2R.predict(X_U2R_test)
pd.crosstab(Y_U2R_test, Y_U2R_pred, rownames=['Actual attacks'], colnames=['Predicted attacks'])

Predicted attacks,0
Actual attacks,
0,9711
4,67


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn import metrics
accuracy = cross_val_score(clf_DoS, X_DoS_test, Y_DoS_test, cv=10, scoring='accuracy')
print("Accuracy: %0.5f (+/- %0.5f)" % (accuracy.mean(), accuracy.std() * 2))
precision = cross_val_score(clf_DoS, X_DoS_test, Y_DoS_test, cv=10, scoring='precision')
print("Precision: %0.5f (+/- %0.5f)" % (precision.mean(), precision.std() * 2))
recall = cross_val_score(clf_DoS, X_DoS_test, Y_DoS_test, cv=10, scoring='recall')
print("Recall: %0.5f (+/- %0.5f)" % (recall.mean(), recall.std() * 2))
f = cross_val_score(clf_DoS, X_DoS_test, Y_DoS_test, cv=10, scoring='f1')
print("F-measure: %0.5f (+/- %0.5f)" % (f.mean(), f.std() * 2))

Accuracy: 0.99796 (+/- 0.00262)
Precision: 0.99906 (+/- 0.00172)
Recall: 0.99678 (+/- 0.00322)
F-measure: 0.99765 (+/- 0.00263)


In [ ]:
accuracy = cross_val_score(clf_Probe, X_Probe_test, Y_Probe_test, cv=10, scoring='accuracy')
print("Accuracy: %0.5f (+/- %0.5f)" % (accuracy.mean(), accuracy.std() * 2))
precision = cross_val_score(clf_Probe, X_Probe_test, Y_Probe_test, cv=10, scoring='precision_macro')
print("Precision: %0.5f (+/- %0.5f)" % (precision.mean(), precision.std() * 2))
recall = cross_val_score(clf_Probe, X_Probe_test, Y_Probe_test, cv=10, scoring='recall_macro')
print("Recall: %0.5f (+/- %0.5f)" % (recall.mean(), recall.std() * 2))
f = cross_val_score(clf_Probe, X_Probe_test, Y_Probe_test, cv=10, scoring='f1_macro')
print("F-measure: %0.5f (+/- %0.5f)" % (f.mean(), f.std() * 2))

Accuracy: 0.99679 (+/- 0.00317)
Precision: 0.99643 (+/- 0.00513)
Recall: 0.99190 (+/- 0.00681)
F-measure: 0.99509 (+/- 0.00488)


In [ ]:
accuracy = cross_val_score(clf_U2R, X_U2R_test, Y_U2R_test, cv=10, scoring='accuracy')
print("Accuracy: %0.5f (+/- %0.5f)" % (accuracy.mean(), accuracy.std() * 2))
precision = cross_val_score(clf_U2R, X_U2R_test, Y_U2R_test, cv=10, scoring='precision_macro')
print("Precision: %0.5f (+/- %0.5f)" % (precision.mean(), precision.std() * 2))
recall = cross_val_score(clf_U2R, X_U2R_test, Y_U2R_test, cv=10, scoring='recall_macro')
print("Recall: %0.5f (+/- %0.5f)" % (recall.mean(), recall.std() * 2))
f = cross_val_score(clf_U2R, X_U2R_test, Y_U2R_test, cv=10, scoring='f1_macro')
print("F-measure: %0.5f (+/- %0.5f)" % (f.mean(), f.std() * 2))

Accuracy: 0.99785 (+/- 0.00213)
Precision: 0.94421 (+/- 0.12175)
Recall: 0.82961 (+/- 0.18582)
F-measure: 0.90080 (+/- 0.09803)


In [ ]:
accuracy = cross_val_score(clf_R2L, X_R2L_test, Y_R2L_test, cv=10, scoring='accuracy')
print("Accuracy: %0.5f (+/- %0.5f)" % (accuracy.mean(), accuracy.std() * 2))
precision = cross_val_score(clf_R2L, X_R2L_test, Y_R2L_test, cv=10, scoring='precision_macro')
print("Precision: %0.5f (+/- %0.5f)" % (precision.mean(), precision.std() * 2))
recall = cross_val_score(clf_R2L, X_R2L_test, Y_R2L_test, cv=10, scoring='recall_macro')
print("Recall: %0.5f (+/- %0.5f)" % (recall.mean(), recall.std() * 2))
f = cross_val_score(clf_R2L, X_R2L_test, Y_R2L_test, cv=10, scoring='f1_macro')
print("F-measure: %0.5f (+/- %0.5f)" % (f.mean(), f.std() * 2))

Accuracy: 0.98007 (+/- 0.00660)
Precision: 0.97377 (+/- 0.00867)
Recall: 0.96961 (+/- 0.01089)
F-measure: 0.97042 (+/- 0.00934)


In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


from google.colab import files
uploaded = files.upload()
uploaded = files.upload()

train = pd.read_csv('NSL_KDD_Train.csv')
test = pd.read_csv('NSL_KDD_Test.csv')


Saving NSL_KDD_Train.csv to NSL_KDD_Train (1).csv


Saving NSL_KDD_Test.csv to NSL_KDD_Test.csv


KeyError: "['label'] not found in axis"

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------------------------
# 1 Define Column Names
# -------------------------
col_names = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
'num_shells','num_access_files','num_outbound_cmds','is_host_login',
'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate','label'
]

# -------------------------
# 2 Load Dataset
# -------------------------
train = pd.read_csv('NSL_KDD_Train.csv', header=None, names=col_names)
test = pd.read_csv('NSL_KDD_Test.csv', header=None, names=col_names)

# -------------------------
# 3 Split Features and Label
# -------------------------
X_train = train.drop('label', axis=1)
y_train = train['label']

X_test = test.drop('label', axis=1)
y_test = test['label']

# -------------------------
# 4 Encode Categorical Features
# -------------------------
le = LabelEncoder()

for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

# Fix label encoding problem
label_encoder = LabelEncoder()

all_labels = pd.concat([y_train, y_test])

label_encoder.fit(all_labels)

y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_test)

# -------------------------
# 5 Feature Scaling
# -------------------------
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------------------------
# 6 Train Logistic Regression
# -------------------------
model = LogisticRegression(max_iter=2000)

model.fit(X_train, y_train)

# -------------------------
# 7 Prediction
# -------------------------
y_pred = model.predict(X_test)

# -------------------------
# 8 Evaluation
# -------------------------
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.7158445706174592

Confusion Matrix
[[  0   0   0 ...   0   0   0]
 [  0 260   0 ...   0   0   0]
 [  0   0   0 ...   0   0   0]
 ...
 [  0   0   0 ...   0   0   0]
 [  0   0   0 ...   0   0   0]
 [  0   0   0 ...   0   0   0]]

Classification Report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       737
           1       0.98      0.72      0.83       359
           2       0.00      0.00      0.00        20
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00      1231
           5       0.00      0.00      0.00       133
           6       0.00      0.00      0.00         1
           7       0.90      0.97      0.94       141
           8       1.00      1.00      1.00         7
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00       293
          11       0.00      0.00      0.00       996
          12       0.00      0.00  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NSL_KDD_Train.csv to NSL_KDD_Train.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NSL_KDD_Test.csv to NSL_KDD_Test.csv


In [ ]:
!pip install matplotlib-venn
!apt-get -qq install -y libfluidsynth1
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
!apt-get -qq install -y graphviz && pip install pydot
!pip install cartopy


E: Package 'libfluidsynth1' has no installation candidate
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/liba/libarchive/libarchive-dev_3.6.0-1ubuntu1.5_amd64.deb  404  Not Found [IP: 91.189.91.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 74.6 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'libarchive'

In [ ]:
  # https://pypi.python.org/pypi/libarchive
  !apt-get -qq update && apt-get -qq install -y libarchive-dev && pip install -U libarchive
  import libarchive

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libarchive13_3.6.0-1ubuntu1.6_amd64.deb ...
Unpacking libarchive13:amd64 (3.6.0-1ubuntu1.6) over (3.6.0-1ubuntu1.5) ...
Selecting previously unselected package libarchive-dev:amd64.
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.6_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Setting up libarchive13:amd64 (3.6.0-1ubuntu1.6) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.s

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Latest Code


In [ ]:
import pandas as pd
import numpy as np
import sys
import io
import random
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
!pip install pyngrok
from pyngrok import ngrok, conf

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

# Install necessary libraries explicitly
!pip install -q streamlit # Install streamlit separately
!pip install -q pyngrok matplotlib-venn pydot cartopy

# Install libarchive with system dependencies
!apt-get -qq update && apt-get -qq install -y libarchive-dev && pip install -U libarchive

# Verify streamlit installation
!streamlit --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.6_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Processing triggers for man-db (2.10.2-1) ...
  Using cached libarchive-0.4.7.tar.gz (23 kB)
  Preparing metadata (setup.py) ... done
  Using cached nose-1.3.7-py3-none-any.whl.metadata (1.7 kB)
Using cached nose-1.3.7-py3-none-any.whl (154 kB)
  Created wheel for libarchive: filename=libarchive-0.4.7-py3-none-any.whl size=31629 sha256=6d94631ad22f1b18a87620d932d6986e7a24dc9e964f87a5a930b2a29fed6db4
  Stored in directory: /root/.cache/pip/wheels/29/20/ab/f101da7b245b996aa097685ef742243725ea61

In [ ]:
# 1. Define Column Names
col_names = ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"]

# 2. Load Dataset
df = pd.read_csv('NSL_KDD_Train.csv', header=None, names=col_names)
df_test = pd.read_csv('NSL_KDD_Test.csv', header=None, names=col_names)

print('Dimensions of the Training set:',df.shape)
print('Dimensions of the Test set:',df_test.shape)

# 3. Identify Categorical Columns
categorical_columns=['protocol_type', 'service', 'flag']

df_categorical_values = df[categorical_columns]
testdf_categorical_values = df_test[categorical_columns]

# 4. Apply Label Encoding to categorical features
df_categorical_values_enc = df_categorical_values.apply(LabelEncoder().fit_transform)
testdf_categorical_values_enc = testdf_categorical_values.apply(LabelEncoder().fit_transform)

# 5. Apply One-Hot Encoding and ensure consistent columns
enc = OneHotEncoder(categories='auto', sparse_output=False, handle_unknown='ignore')

df_categorical_values_ohe = enc.fit_transform(df_categorical_values_enc)
ohe_feature_names = enc.get_feature_names_out(categorical_columns)

df_cat_data = pd.DataFrame(df_categorical_values_ohe, columns=ohe_feature_names)

testdf_categorical_values_ohe = enc.transform(testdf_categorical_values_enc)
testdf_cat_data = pd.DataFrame(testdf_categorical_values_ohe, columns=ohe_feature_names)

# Re-initialize newdf and newdf_test by joining categorical data
newdf = df.join(df_cat_data)
newdf.drop('flag', axis=1, inplace=True)
newdf.drop('protocol_type', axis=1, inplace=True)
newdf.drop('service', axis=1, inplace=True)

# test data
newdf_test = df_test.join(testdf_cat_data)
newdf_test.drop('flag', axis=1, inplace=True)
newdf_test.drop('protocol_type', axis=1, inplace=True)
newdf_test.drop('service', axis=1, inplace=True)

print('Dimensions of newdf:', newdf.shape)
print('Dimensions of newdf_test:', newdf_test.shape)

# Re-apply label mapping from cell zV4LI5dso0tt
labeldf = newdf['label']
labeldf_test = newdf_test['label']

newlabeldf = labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})
newlabeldf_test = labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})

newdf['label'] = newlabeldf
newdf_test['label'] = newlabeldf_test

# Dictionary to map numerical labels (0-4) back to descriptive string labels
label_to_category_map = {
    0: 'normal',
    1: 'DoS',
    2: 'Probe',
    3: 'R2L',
    4: 'U2R'
}

# Create 'labels5' column (5-category string labels)
newdf['labels5'] = newdf['label'].map(label_to_category_map)
newdf_test['labels5'] = newdf_test['label'].map(label_to_category_map)

# Create 'labels2' column for binary classification (normal vs attack)
newdf['labels2'] = newdf['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
newdf_test['labels2'] = newdf_test['labels5'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print('Distribution of labels2 in Training set:')
print(newdf['labels2'].value_counts())
print('\nDistribution of labels2 in Test set:')
print(newdf_test['labels2'].value_counts())

print('\nDistribution of labels5 in Training set:')
print(newdf['labels5'].value_counts())
print('\nDistribution of labels5 in Test set:')
print(newdf_test['labels5'].value_counts())

# Label Encoding for new label columns
le_labels2 = LabelEncoder()
le_labels5 = LabelEncoder()

all_labels2 = pd.concat([newdf['labels2'], newdf_test['labels2']]).unique()
le_labels2.fit(all_labels2)

newdf['labels2_index'] = le_labels2.transform(newdf['labels2'])
newdf_test['labels2_index'] = le_labels2.transform(newdf_test['labels2'])

all_labels5 = pd.concat([newdf['labels5'], newdf_test['labels5']]).unique()
le_labels5.fit(all_labels5)

newdf['labels5_index'] = le_labels5.transform(newdf['labels5'])
newdf_test['labels5_index'] = le_labels5.transform(newdf_test['labels5'])

print('\nLabels2 Index Mapping:')
for i, label in enumerate(le_labels2.classes_):
    print(f'{label}: {i}')

print('\nLabels5 Index Mapping:')
for i, label in enumerate(le_labels5.classes_):
    print(f'{label}: {i}')

# Splitting into attack types
to_drop_DoS = [0,1]
to_drop_Probe = [0,2]
to_drop_R2L = [0,3]
to_drop_U2R = [0,4]


DoS_df=newdf[newdf['label'].isin(to_drop_DoS)];
Probe_df=newdf[newdf['label'].isin(to_drop_Probe)];
R2L_df=newdf[newdf['label'].isin(to_drop_R2L)];
U2R_df=newdf[newdf['label'].isin(to_drop_U2R)];


#test
DoS_df_test=newdf_test[newdf_test['label'].isin(to_drop_DoS)];
Probe_df_test=newdf_test[newdf_test['label'].isin(to_drop_Probe)];
R2L_df_test=newdf_test[newdf_test['label'].isin(to_drop_R2L)];
U2R_df_test=newdf_test[newdf_test['label'].isin(to_drop_U2R)];


print('\nTrain:')
print('Dimensions of DoS:' ,DoS_df.shape)
print('Dimensions of Probe:' ,Probe_df.shape)
print('Dimensions of R2L:' ,R2L_df.shape)
print('Dimensions of U2R:' ,U2R_df.shape)
print()
print('Test:')
print('Dimensions of DoS:' ,DoS_df_test.shape)
print('Dimensions of Probe:' ,Probe_df_test.shape)
print('Dimensions of R2L:' ,R2L_df_test.shape)
print('Dimensions of U2R:' ,U2R_df_test.shape)

Dimensions of the Training set: (125973, 42)
Dimensions of the Test set: (22544, 42)
Dimensions of newdf: (125973, 123)
Dimensions of newdf_test: (22544, 123)


/tmp/ipykernel_29811/3037608178.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf = labeldf.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
/tmp/ipykernel_29811/3037608178.py:64: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf_test = labeldf_test.replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1

Distribution of labels2 in Training set:
labels2
normal    67343
attack    58630
Name: count, dtype: int64

Distribution of labels2 in Test set:
labels2
attack    12833
normal     9711
Name: count, dtype: int64

Distribution of labels5 in Training set:
labels5
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Distribution of labels5 in Test set:
labels5
normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64

Labels2 Index Mapping:
attack: 0
normal: 1

Labels5 Index Mapping:
DoS: 0
Probe: 1
R2L: 2
U2R: 3
normal: 4

Train:
Dimensions of DoS: (113270, 127)
Dimensions of Probe: (78999, 127)
Dimensions of R2L: (68338, 127)
Dimensions of U2R: (67395, 127)

Test:
Dimensions of DoS: (17171, 127)
Dimensions of Probe: (12132, 127)
Dimensions of R2L: (12596, 127)
Dimensions of U2R: (9778, 127)


In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle


def load_dataset():
    train_path = '/content/NSL_KDD_Train.csv'
    test_path = '/content/NSL_KDD_Test.csv'

    if not os.path.exists(train_path):
        raise FileNotFoundError(
            f"❌ NSL_KDD_Train.csv not found at {train_path}!"
            "➡️ Ensure the file is uploaded to the `/content/` directory."
        )

    if not os.path.exists(test_path):
        raise FileNotFoundError(
            f"❌ NSL_KDD_Test.csv not found at {test_path}!"
            "➡️ Ensure the file is uploaded to the `/content/` directory."
        )

    print(f"✅ Loading train from: {train_path}")
    print(f"✅ Loading test from: {test_path}")

    train = pd.read_csv(train_path, header=None)
    test = pd.read_csv(test_path, header=None)

    return train, test

# Load dataset safely
train, test = load_dataset()

# Combine datasets
data = pd.concat([train, test], ignore_index=True)

# Provide column names as the original dataframes loaded without headers
col_names = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','label'
]
data.columns = col_names

print("Columns:", data.columns)

# Detect label column
label_col = 'label'

# Convert labels to binary
# Ensure 'normal' is mapped to 'Normal' and others to 'Attack'
data[label_col] = data[label_col].apply(lambda x: 'Normal' if x == 'normal' else 'Attack')

# Encode categorical columns
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
if label_col in categorical_cols:
    categorical_cols.remove(label_col)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

# Features & target
X = data.drop(label_col, axis=1)
y = data[label_col]

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
report = classification_report(y_test, preds)
print(report)

# Save evaluation metrics
metrics = {
    'accuracy': accuracy_score(y_test, preds),
    'classification_report': classification_report(y_test, preds, output_dict=True)
}
pickle.dump(metrics, open('model_metrics.pkl', 'wb'))
print("✅ Model metrics saved to model_metrics.pkl")

# Save artifacts
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(X.shape[1], open('feature_count.pkl', 'wb'))
pickle.dump(encoders, open('encoders.pkl', 'wb')) # Save the encoders
pickle.dump(col_names, open('col_names.pkl', 'wb')) # Save original column names
pickle.dump(X.columns.tolist(), open('feature_names_rf.pkl', 'wb')) # Save actual feature names for RF


# ================= BASIC TEST =================

# Run this after training to verify everything works
if __name__ == "__main__":
    print("\n✅ Running basic test...")
    sample = np.zeros((1, X.shape[1]))
    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)
    print("Sample Prediction:", pred)


✅ Loading train from: /content/NSL_KDD_Train.csv
✅ Loading test from: /content/NSL_KDD_Test.csv
Columns: Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label'],
      dtype='object')
Accuracy: 0.9956908160517102
              p

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
print("--- Training 5-Class Random Forest Model ---")

# Features for 5-class classification
# Drop all label-related columns to get the feature set
features_to_drop = ['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']
X_5class_train = newdf.drop(columns=features_to_drop)
y_5class_train = newdf['labels5_index']

X_5class_test = newdf_test.drop(columns=features_to_drop)
y_5class_test = newdf_test['labels5_index']

print("Shape of X_5class_train:", X_5class_train.shape)
print("Shape of y_5class_train:", y_5class_train.shape)
print("Shape of X_5class_test:", X_5class_test.shape)
print("Shape of y_5class_test:", y_5class_test.shape)

# Scale features
scaler_5class = StandardScaler()
X_5class_train_scaled = scaler_5class.fit_transform(X_5class_train)
X_5class_test_scaled = scaler_5class.transform(X_5class_test)

# Train the 5-class model with class_weight='balanced'
model_5class = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
model_5class.fit(X_5class_train_scaled, y_5class_train)

# Evaluate the 5-class model
preds_5class = model_5class.predict(X_5class_test_scaled)
accuracy_5class = accuracy_score(y_5class_test, preds_5class)
report_5class = classification_report(y_5class_test, preds_5class, output_dict=True)

print("\nAccuracy (5-class model):", accuracy_5class)
print("Classification Report (5-class model):\n", classification_report(y_5class_test, preds_5class))

# Save 5-class model artifacts
pickle.dump(model_5class, open('model_5class.pkl', 'wb'))
pickle.dump(scaler_5class, open('scaler_5class.pkl', 'wb'))
pickle.dump(X_5class_train.columns.tolist(), open('feature_names_5class.pkl', 'wb')) # Save actual feature names for 5-class RF
pickle.dump(le_labels5, open('le_labels5.pkl', 'wb')) # Save the LabelEncoder for 5-class labels
pickle.dump({'accuracy': accuracy_5class, 'classification_report': report_5class}, open('model_5class_metrics.pkl', 'wb'))

print("✅ 5-Class Random Forest model and artifacts saved.")

--- Training 5-Class Random Forest Model ---
Shape of X_5class_train: (125973, 122)
Shape of y_5class_train: (125973,)
Shape of X_5class_test: (22544, 122)
Shape of y_5class_test: (22544,)

Accuracy (5-class model): 0.7025816181689141
Classification Report (5-class model):
               precision    recall  f1-score   support

           0       0.96      0.66      0.78      7460
           1       0.84      0.59      0.69      2421
           2       1.00      0.00      0.00      2885
           3       0.00      0.00      0.00        67
           4       0.60      0.98      0.75      9711

    accuracy                           0.70     22544
   macro avg       0.68      0.45      0.44     22544
weighted avg       0.80      0.70      0.65     22544

✅ 5-Class Random Forest model and artifacts saved.


### Training a Simple Binary Classification Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import pickle # Added to save models

print("--- Training Binary Logistic Regression Model ---")

# Features and target for binary classification
# Drop all label-related columns for the feature set
features_to_drop_lr = ['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']
X_train_lr = newdf.drop(columns=features_to_drop_lr)
Y_train_lr = newdf['labels2_index'] # Use labels2_index for binary classification

X_test_lr = newdf_test.drop(columns=features_to_drop_lr)
Y_test_lr = newdf_test['labels2_index'] # Use labels2_index for binary classification

print("Shape of X_train_lr:", X_train_lr.shape)
print("Shape of Y_train_lr:", Y_train_lr.shape)
print("Shape of X_test_lr:", X_test_lr.shape)
print("Shape of Y_test_lr:", Y_test_lr.shape)

# Scale features
scaler_lr = StandardScaler()
X_train_lr_scaled = scaler_lr.fit_transform(X_train_lr)
X_test_lr_scaled = scaler_lr.transform(X_test_lr)

# Train the Logistic Regression model
model_lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1) # Increased max_iter for convergence
model_lr.fit(X_train_lr_scaled, Y_train_lr)

# Evaluate the model
preds_lr = model_lr.predict(X_test_lr_scaled)
accuracy_lr = accuracy_score(Y_test_lr, preds_lr)
report_lr = classification_report(Y_test_lr, preds_lr)
conf_matrix_lr = confusion_matrix(Y_test_lr, preds_lr)

print("\nAccuracy (Logistic Regression):", accuracy_lr)
print("\nClassification Report (Logistic Regression):\n", report_lr)
print("\nConfusion Matrix (Logistic Regression):\n", conf_matrix_lr)

# Save Logistic Regression model artifacts
pickle.dump(model_lr, open('model_lr.pkl', 'wb'))
pickle.dump(scaler_lr, open('scaler_lr.pkl', 'wb'))
pickle.dump(X_train_lr.columns.tolist(), open('feature_names_lr.pkl', 'wb')) # Save actual feature names for LR
pickle.dump({'accuracy': accuracy_lr, 'classification_report': classification_report(Y_test_lr, preds_lr, output_dict=True), 'confusion_matrix': conf_matrix_lr.tolist()}, open('model_lr_metrics.pkl', 'wb'))

print("✅ Binary Logistic Regression model and artifacts saved.")

--- Training Binary Logistic Regression Model ---
Shape of X_train_lr: (125973, 122)
Shape of Y_train_lr: (125973,)
Shape of X_test_lr: (22544, 122)
Shape of Y_test_lr: (22544,)

Accuracy (Logistic Regression): 0.6678495386799148

Classification Report (Logistic Regression):
               precision    recall  f1-score   support

           0       0.67      0.83      0.74     12833
           1       0.67      0.45      0.54      9711

    accuracy                           0.67     22544
   macro avg       0.67      0.64      0.64     22544
weighted avg       0.67      0.67      0.65     22544


Confusion Matrix (Logistic Regression):
 [[10667  2166]
 [ 5322  4389]]
✅ Binary Logistic Regression model and artifacts saved.


In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler # Ensure all are imported
from sklearn.svm import LinearSVC

st.set_page_config(page_title="IDS Dashboard", layout="wide")

st.title("🔐 Intelligent IDS Dashboard")
st.markdown("### Real-time Network Traffic Detection")

# --- Model Selection (Hardcoded to Binary Random Forest for primary interaction) ---
# Allow selection for displaying metrics, but primary prediction will be RF Binary
st.sidebar.header("Model Selection for Primary Prediction")
selected_prediction_model = st.sidebar.selectbox(
    "Choose a model for single/batch predictions:",
    ["Binary (Normal/Attack) Random Forest"] # Only one for now as per the previous app.py logic
)

st.sidebar.markdown("---")

# Initialize variables for all models and their artifacts
model_rf_binary = None
scaler_rf_binary = None
encoders_rf_binary = None
feature_names_rf_binary = None
metrics_rf_binary = None

model_rf_5class = None
scaler_rf_5class = None
feature_names_rf_5class = None
le_labels5 = None
metrics_rf_5class = None

model_svm_binary = None
scaler_svm_binary = None
feature_names_svm_binary = None
metrics_svm_binary = None

col_names_original = None # Original raw column names

# Load ALL model artifacts
try:
    # Binary Random Forest
    model_rf_binary = pickle.load(open('/content/model.pkl', 'rb'))
    scaler_rf_binary = pickle.load(open('/content/scaler.pkl', 'rb'))
    encoders_rf_binary = pickle.load(open('/content/encoders.pkl', 'rb'))
    col_names_original = pickle.load(open('/content/col_names.pkl', 'rb'))
    feature_names_rf_binary = pickle.load(open('/content/feature_names_rf.pkl', 'rb'))
    metrics_rf_binary = pickle.load(open('/content/model_metrics.pkl', 'rb'))

    # 5-Class Random Forest
    model_rf_5class = pickle.load(open('/content/model_5class.pkl', 'rb'))
    scaler_rf_5class = pickle.load(open('/content/scaler_5class.pkl', 'rb'))
    feature_names_rf_5class = pickle.load(open('/content/feature_names_5class.pkl', 'rb'))
    le_labels5 = pickle.load(open('/content/le_labels5.pkl', 'rb'))
    metrics_rf_5class = pickle.load(open('/content/model_5class_metrics.pkl', 'rb'))

    # Binary SVM
    model_svm_binary = pickle.load(open('/content/svm_model.pkl', 'rb'))
    scaler_svm_binary = pickle.load(open('/content/svm_scaler.pkl', 'rb'))
    feature_names_svm_binary = pickle.load(open('/content/feature_names_svm.pkl', 'rb'))
    metrics_svm_binary = pickle.load(open('/content/svm_model_metrics.pkl', 'rb'))


    st.sidebar.success("✅ All models and artifacts loaded successfully!")
except FileNotFoundError as e:
    st.error(f"❌ Model or preprocessing files not found: {e}. Please ensure the training script has been run and all necessary files exist in /content/")
    st.stop()
except Exception as e:
    st.error(f"❌ Failed to load model or its artifacts: {e}")
    import traceback
    st.error(traceback.format_exc())
    st.stop()

st.sidebar.header("Input Features")

# --- Function to preprocess data ---
def preprocess_input(df_raw, original_col_names, feature_names_model, encoders, scaler):
    processed_df = df_raw.copy()

    # Ensure all original columns are present, fill missing with 0 or appropriate default
    for col in [c for c in original_col_names if c != 'label']:
        if col not in processed_df.columns:
            processed_df[col] = 0 # Default value for missing columns in raw input

    # Reorder columns to match the training data
    processed_df = processed_df[[c for c in original_col_names if c != 'label']] # Exclude 'label'

    categorical_cols_to_encode = ['protocol_type', 'service', 'flag']

    # Apply LabelEncoders to input_df_raw to get integer-encoded categorical values
    temp_le_df = pd.DataFrame(index=df_raw.index)
    for col in categorical_cols_to_encode:
        if col in df_raw.columns and col in encoders:
            le = encoders[col]
            # Ensure unseen categories are handled by the LabelEncoder; map to a placeholder
            # Original ValueError: operands could not be broadcast together with shapes (1,) (4,) fixed here by [0]
            temp_le_df[col] = processed_df[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
            temp_le_df[col] = temp_le_df[col].astype(int)
        else:
            # If a categorical column is missing in the input, fill with a placeholder
            temp_le_df[col] = -1
            processed_df[col] = -1 # Also update processed_df for later column drops if needed

    # Reconstruct the OneHotEncoder using the learned label-encoded categories
    ohe_categories_for_fit = []
    for col in categorical_cols_to_encode:
        if col in encoders:
            ohe_categories_for_fit.append(np.arange(len(encoders[col].classes_)))

    ohe_for_inference = OneHotEncoder(categories=ohe_categories_for_fit, sparse_output=False, handle_unknown='ignore')

    # Fit the OneHotEncoder on a dummy input to generate feature names
    dummy_input_for_ohe_fit = pd.DataFrame([[0] * len(categorical_cols_to_encode)], columns=categorical_cols_to_encode)
    ohe_for_inference.fit(dummy_input_for_ohe_fit)

    # Now transform the actual label-encoded input data
    df_categorical_values_ohe = ohe_for_inference.transform(temp_le_df)
    ohe_feature_names = ohe_for_inference.get_feature_names_out(categorical_cols_to_encode)
    df_cat_data = pd.DataFrame(df_categorical_values_ohe, columns=ohe_feature_names, index=processed_df.index)

    # Drop original categorical columns from processed_df
    processed_df = processed_df.drop(columns=categorical_cols_to_encode)

    # Join OHE features
    processed_df = processed_df.reset_index(drop=True).join(df_cat_data.reset_index(drop=True))

    # Ensure all columns are numeric
    for col in processed_df.columns:
        processed_df.loc[:, col] = pd.to_numeric(processed_df[col], errors='coerce')
        processed_df.loc[:, col] = processed_df[col].fillna(0) # Fill NaNs if any coercion failed

    # Align columns with the training feature set (`feature_names_model`)
    # Add missing columns (from training set) with 0, and drop extra columns (from input)
    final_processed_df = pd.DataFrame(0, index=processed_df.index, columns=feature_names_model)
    for col in feature_names_model:
        if col in processed_df.columns:
            final_processed_df[col] = processed_df[col]

    # Scale the processed data
    scaled_input_df = scaler.transform(final_processed_df)
    return scaled_input_df

# Option to upload a CSV or TXT file for batch prediction
uploaded_file = st.sidebar.file_uploader("Upload a CSV or TXT file for batch prediction", type=["csv", "txt"])

if uploaded_file is not None:
    st.subheader("Uploaded Data Preview")
    try:
        if uploaded_file.name.endswith('.csv'):
            input_df_raw = pd.read_csv(uploaded_file, header=None, names=[c for c in col_names_original if c != 'label'])
        elif uploaded_file.name.endswith('.txt'):
            input_df_raw = pd.read_csv(uploaded_file, header=None, names=[c for c in col_names_original if c != 'label'])
        else:
            st.error("Unsupported file type. Please upload a CSV or TXT file.")
            st.stop()

        st.write(input_df_raw.head())

        # --- Binary Random Forest Prediction for Uploaded Data ---
        # Preprocess the uploaded data for Binary Random Forest prediction
        scaled_input_df_binary_rf = preprocess_input(input_df_raw.copy(), col_names_original, feature_names_rf_binary, encoders_rf_binary, scaler_rf_binary)

        # Make predictions using the primary binary RF model
        predictions_binary_rf = model_rf_binary.predict(scaled_input_df_binary_rf)

        st.subheader("Batch Prediction Results (Binary Random Forest)")
        input_df_raw['binary_prediction'] = predictions_binary_rf

        attack_data_binary = input_df_raw[input_df_raw['binary_prediction'] == 'Attack']
        if not attack_data_binary.empty:
            st.error("⚠️ Attacks Detected (Binary Model)!")
            st.write("### List of Attacker Details (Binary Model):")
            st.dataframe(attack_data_binary)
            st.write(f"Total Attacks (Binary Model): {len(attack_data_binary)}")
        else:
            st.success("✅ No Attacks Detected (Binary Model) in the uploaded data.")

        # Visualize Binary Prediction Distribution
        st.subheader("Predicted Traffic Distribution (Normal/Attack - Binary RF)")
        prediction_counts_binary = input_df_raw['binary_prediction'].value_counts()
        fig_batch_pred_binary, ax_batch_pred_binary = plt.subplots(figsize=(8, 5))
        sns.barplot(x=prediction_counts_binary.index, y=prediction_counts_binary.values, ax=ax_batch_pred_binary, palette='viridis')
        ax_batch_pred_binary.set_title('Distribution of Predicted Traffic Types (Normal/Attack)')
        ax_batch_pred_binary.set_xlabel('Prediction')
        ax_batch_pred_binary.set_ylabel('Count')
        st.pyplot(fig_batch_pred_binary)
        plt.close(fig_batch_pred_binary)

        st.markdown("--- ")

        # --- 5-Class Prediction for Uploaded Data ---
        st.subheader("Batch Prediction Results (5-Class Random Forest)")
        # Preprocess the uploaded data for 5-class Random Forest prediction
        # Use the scaler and feature names specifically for the 5-class model
        scaled_input_df_5class_rf = preprocess_input(input_df_raw.copy(), col_names_original, feature_names_rf_5class, encoders_rf_binary, scaler_rf_5class)

        # Make predictions using the 5-class RF model
        predictions_5class_rf_indices = model_rf_5class.predict(scaled_input_df_5class_rf)
        # Convert numerical predictions back to descriptive string labels
        predictions_5class_rf_labels = le_labels5.inverse_transform(predictions_5class_rf_indices)

        input_df_raw['5class_prediction'] = predictions_5class_rf_labels

        st.write("### Predicted 5-Class Categories for Uploaded Data:")
        st.dataframe(input_df_raw[['binary_prediction', '5class_prediction']].head()) # Show both predictions

        # Visualize 5-Class Prediction Distribution
        st.subheader("Predicted Traffic Distribution (5-Class - RF)")
        prediction_counts_5class = input_df_raw['5class_prediction'].value_counts()
        fig_batch_pred_5class, ax_batch_pred_5class = plt.subplots(figsize=(10, 6))
        sns.barplot(x=prediction_counts_5class.index, y=prediction_counts_5class.values, ax=ax_batch_pred_5class, palette='magma')
        ax_batch_pred_5class.set_title('Distribution of Predicted Traffic Types (5 Categories)')
        ax_batch_pred_5class.set_xlabel('Prediction Category')
        ax_batch_pred_5class.set_ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        st.pyplot(fig_batch_pred_5class)
        plt.close(fig_batch_pred_5class)

    except Exception as e:
        st.error(f"Error processing uploaded file: {e}")
        import traceback
        st.error(traceback.format_exc())

# Inputs for single prediction (using Binary Random Forest for prediction)
st.subheader("Manual Single Traffic Prediction")
manual_inputs = {}
for col in [c for c in col_names_original if c not in ['label', 'protocol_type', 'service', 'flag']]:
    if col in ['duration', 'src_bytes', 'dst_bytes', 'count']: # Example numerical fields
        manual_inputs[col] = st.sidebar.number_input(col.replace('_', ' ').title(), value=0, key=f'manual_{col}')
    elif col in ['serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']:
         manual_inputs[col] = st.sidebar.number_input(col.replace('_', ' ').title(), value=0.0, format="%.6f", key=f'manual_{col}')
    else: # Other numerical, often binary or small integers
         manual_inputs[col] = st.sidebar.number_input(col.replace('_', ' ').title(), value=0, key=f'manual_{col}')


# Dynamic categorical inputs - must match encoding logic
categorical_cols_for_single = ['protocol_type', 'service', 'flag']
manual_inputs['protocol_type'] = st.sidebar.selectbox("Protocol Type", encoders_rf_binary['protocol_type'].classes_, key='manual_protocol_type')
manual_inputs['service'] = st.sidebar.selectbox("Service", encoders_rf_binary['service'].classes_, key='manual_service')
manual_inputs['flag'] = st.sidebar.selectbox("Flag", encoders_rf_binary['flag'].classes_, key='manual_flag')


if st.sidebar.button("🚀 Analyze Single Traffic"):
    try:
        # Create a raw DataFrame from manual inputs
        single_input_df_raw = pd.DataFrame([manual_inputs])

        # Preprocess the single input using the same function
        scaled_single_input = preprocess_input(single_input_df_raw, col_names_original, feature_names_rf_binary, encoders_rf_binary, scaler_rf_binary)

        # Make a prediction for the single input using the primary binary RF model
        single_prediction = model_rf_binary.predict(scaled_single_input)

        st.subheader("Single Traffic Prediction Output (Binary Random Forest)")
        display_prediction = single_prediction[0]

        if display_prediction == 'Attack':
            st.error(f"⚠️ {display_prediction} Detected!")
        else:
            st.success(f"✅ {display_prediction} Traffic")

    except Exception as e:
        st.error(f"Error during single prediction: {e}")
        import traceback
        st.error(traceback.format_exc())


st.markdown("---")

# --- Model Explanation: Feature Importances (Binary Random Forest) ---
st.subheader("💡 Understanding the Model's Decisions (Binary Random Forest)")
if model_rf_binary and hasattr(model_rf_binary, 'feature_importances_') and feature_names_rf_binary:
    if len(feature_names_rf_binary) == len(model_rf_binary.feature_importances_):
        importance_df = pd.DataFrame({
            'Feature': feature_names_rf_binary,
            'Importance': model_rf_binary.feature_importances_
        })
        importance_df = importance_df.sort_values(by='Importance', ascending=False).head(15) # Top 15 features

        fig_feat_imp, ax_feat_imp = plt.subplots(figsize=(10, 7))
        sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis', ax=ax_feat_imp)
        ax_feat_imp.set_title('Top 15 Most Important Features for Binary Random Forest')
        ax_feat_imp.set_xlabel('Feature Importance')
        ax_feat_imp.set_ylabel('Feature')
        st.pyplot(fig_feat_imp)
        plt.close(fig_feat_imp)
    else:
        st.warning("Could not match feature importances with column names for Binary RF. Number of features in model does not match provided feature names.")
else:
    st.info("Binary Random Forest model or feature importances not available to display.")

st.markdown("---")

# --- Display Evaluation Metrics for ALL Models ---
st.header("📊 Model Performance Metrics")

# Binary Random Forest Metrics
if metrics_rf_binary:
    st.subheader("1. Binary Random Forest Model Metrics")
    st.write(f"**Overall Accuracy:** {metrics_rf_binary['accuracy']:.4f}")
    st.write("### Detailed Classification Report")
    report_rf_binary_df = pd.DataFrame(metrics_rf_binary['classification_report']).transpose()
    st.dataframe(report_rf_binary_df.style.format({'precision': '{:.2f}', 'recall': '{:.2f}', 'f1-score': '{:.2f}'}))
else:
    st.warning("Binary Random Forest metrics not available.")

st.markdown("---")

# 5-Class Random Forest Metrics and Distribution
if metrics_rf_5class and le_labels5:
    st.subheader("2. 5-Class Random Forest Model Metrics")
    st.write(f"**Overall Accuracy:** {metrics_rf_5class['accuracy']:.4f}")
    st.write("### Detailed Classification Report")
    report_rf_5class_df = pd.DataFrame(metrics_rf_5class['classification_report']).transpose()
    st.dataframe(report_rf_5class_df.style.format({'precision': '{:.2f}', 'recall': '{:.2f}', 'f1-score': '{:.2f}'}))

    st.subheader("🌐 Dataset Attack Category Distribution (5-Class Model Perspective)")
    st.markdown(
        """
        This section visualizes the distribution of network traffic across 5 distinct attack categories (DoS, Probe, R2L, U2R, Normal),
        as identified by the 5-class classification model trained on the dataset.
        It provides insight into the types of threats present in the data.
        """
    )
    # Get support/counts from the 5-class classification report
    class_supports = {}
    for i, class_name_encoded in enumerate(le_labels5.classes_):
        # The classification report's keys are strings of the numeric labels
        numeric_label_key = str(i)
        if numeric_label_key in metrics_rf_5class['classification_report']:
            class_supports[class_name_encoded] = metrics_rf_5class['classification_report'][numeric_label_key]['support']
        else:
            class_supports[class_name_encoded] = 0 # Handle cases where a label might not have support

    five_class_dist_df = pd.DataFrame(list(class_supports.items()), columns=['Category', 'Count'])
    five_class_dist_df = five_class_dist_df.sort_values(by='Count', ascending=False)

    fig_5class_dist, ax_5class_dist = plt.subplots(figsize=(10, 6))
    sns.barplot(x='Category', y='Count', data=five_class_dist_df, palette='magma', ax=ax_5class_dist)
    ax_5class_dist.set_title('Distribution of 5 Attack Categories (from Dataset/5-Class Model)')
    ax_5class_dist.set_xlabel('Attack Category')
    ax_5class_dist.set_ylabel('Number of Samples')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    st.pyplot(fig_5class_dist)
    plt.close(fig_5class_dist)
else:
    st.warning("5-Class Random Forest metrics or label encoder not available.")

st.markdown("---")

# Binary SVM Metrics
if metrics_svm_binary:
    st.subheader("3. Binary SVM Model Metrics")
    st.write(f"**Overall Accuracy:** {metrics_svm_binary['accuracy']:.4f}")
    st.write("### Detailed Classification Report")
    report_svm_binary_df = pd.DataFrame(metrics_svm_binary['classification_report']).transpose()
    st.dataframe(report_svm_binary_df.style.format({'precision': '{:.2f}', 'recall': '{:.2f}', 'f1-score': '{:.2f}'}))
else:
    st.warning("Binary SVM metrics not available.")

st.markdown("---")

st.markdown("**Note:** To apply changes to the Streamlit app, you need to restart the app (e.g., by re-running the ngrok cell).")

Overwriting app.py


In [ ]:
import os
import time
from pyngrok import ngrok, conf

# Get ngrok authtoken from user input
try:
    # Check if NGROK_AUTH_TOKEN is already set as an environment variable
    if "NGROK_AUTH_TOKEN" not in os.environ or not os.environ["NGROK_AUTH_TOKEN"]:
        ngrok_auth_token = input("Enter your ngrok Authtoken: ")
        os.environ["NGROK_AUTH_TOKEN"] = ngrok_auth_token

    # Configure ngrok
    ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])

    # Kill any existing ngrok processes and processes on port 8501
    ngrok.kill()
    !fuser -k 8501/tcp # Kill processes using port 8501

    print("Starting Streamlit app...")
    # Run the Streamlit app in the background and log its output
    streamlit_log_file = "streamlit.log"
    # Use 'python -m streamlit' to ensure the correct executable is found
    !python -m streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats=False --server.headless=True > {streamlit_log_file} 2>&1 &

    # Give Streamlit some time to start up
    time.sleep(10) # Increased sleep to 10 seconds to give it more time to write logs

    # Check if Streamlit process is running
    print("\n--- Streamlit Process Status ---")
    !ps aux | grep streamlit

    # Print Streamlit logs
    print(f"\n--- Streamlit Logs from {streamlit_log_file} ---")
    if os.path.exists(streamlit_log_file) and os.path.getsize(streamlit_log_file) > 0:
        with open(streamlit_log_file, 'r') as f:
            print(f.read())
    else:
        print("No Streamlit logs generated yet, or file is empty.")

    # Start a Streamlit app and expose it via ngrok
    # Streamlit runs on port 8501 by default
    public_url = ngrok.connect(8501)
    print(f"Your Streamlit app is live at: {public_url}")

    print("Streamlit app is running. Access it using the public URL above.")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure your ngrok Authtoken is correct and try again.")

8501/tcp:            38101
Starting Streamlit app...

--- Streamlit Process Status ---
root       41260 22.3  0.5 525296 70980 ?        Sl   13:36   0:02 python3 -m streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats=False --server.headless=True
root       41309  0.0  0.0   7372  3456 ?        S    13:36   0:00 /bin/bash -c ps aux | grep streamlit
root       41311  0.0  0.0   6480  2368 ?        S    13:36   0:00 grep streamlit

--- Streamlit Logs from streamlit.log ---
2026-05-18 13:36:12.279 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.231.51.132:8501


Your Streamlit app is live at: NgrokTunnel: "https://clean-justify-convene.ngrok-free.dev" -> "http://localhost:8501"
Streamlit app is running. Access it using the public URL above.


In [ ]:
print("--- Preparing data for Binary SVM Classification ---")

# Features for Binary Classification (Normal vs. Attack)
# Drop all label-related columns for the feature set
features_to_drop_svm = ['label', 'labels2', 'labels2_index', 'labels5', 'labels5_index']
X_train_svm = newdf.drop(columns=features_to_drop_svm)
Y_train_svm = newdf['labels2_index'] # Use labels2_index for binary classification

X_test_svm = newdf_test.drop(columns=features_to_drop_svm)
Y_test_svm = newdf_test['labels2_index'] # Use labels2_index for binary classification

print("Shape of X_train_svm:", X_train_svm.shape)
print("Shape of Y_train_svm:", Y_train_svm.shape)
print("Shape of X_test_svm:", X_test_svm.shape)
print("Shape of Y_test_svm:", Y_test_svm.shape)

# Save feature names used for SVM model training (after dropping labels)
pickle.dump(X_train_svm.columns.tolist(), open('feature_names_svm.pkl', 'wb'))

--- Preparing data for Binary SVM Classification ---
Shape of X_train_svm: (125973, 122)
Shape of Y_train_svm: (125973,)
Shape of X_test_svm: (22544, 122)
Shape of Y_test_svm: (22544,)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import pickle
import pandas as pd

# Assume X_train_svm, Y_train_svm, X_test_svm, Y_test_svm are already defined from previous cells.

print("-- Handling NaN values and Scaling SVM Features ---")

# Fill any NaN values before scaling
# A common strategy for network data is to fill NaNs with 0, or with the mean/median of the column.
# For this dataset, we'll fill with 0, assuming missing values imply absence of that feature's activity.
X_train_svm = X_train_svm.fillna(0)
X_test_svm = X_test_svm.fillna(0)

# Initialize a StandardScaler for SVM features
svm_scaler = StandardScaler()

# Fit on training data and transform both training and testing data
X_train_svm_scaled = svm_scaler.fit_transform(X_train_svm)
X_test_svm_scaled = svm_scaler.transform(X_test_svm)

print("Shape of X_train_svm_scaled:", X_train_svm_scaled.shape)
print("Shape of X_test_svm_scaled:", X_test_svm_scaled.shape)

print("\n--- Training SVM Model (LinearSVC) ---")
# Initialize LinearSVC model. For large datasets, LinearSVC is often faster than SVC(kernel='linear')
# max_iter is increased to ensure convergence for potentially complex datasets
svm_model = LinearSVC(max_iter=5000, random_state=42)

# Train the model
svm_model.fit(X_train_svm_scaled, Y_train_svm)

print("\n--- Evaluating SVM Model ---")
# Make predictions on the test set
svm_preds = svm_model.predict(X_test_svm_scaled)

# Calculate accuracy
svm_accuracy = accuracy_score(Y_test_svm, svm_preds)
print("SVM Accuracy:", svm_accuracy)

# Generate classification report
svm_report = classification_report(Y_test_svm, svm_preds)
print("SVM Classification Report:\n", svm_report)

# Save evaluation metrics for SVM
svm_metrics = {
    'accuracy': svm_accuracy,
    'classification_report': classification_report(Y_test_svm, svm_preds, output_dict=True)
}
pickle.dump(svm_metrics, open('svm_model_metrics.pkl', 'wb'))
print("\u2705 SVM Model metrics saved to svm_model_metrics.pkl")

# Save the trained SVM model and its scaler
pickle.dump(svm_model, open('svm_model.pkl', 'wb'))
pickle.dump(svm_scaler, open('svm_scaler.pkl', 'wb'))
print("\u2705 SVM Model and Scaler saved.")

-- Handling NaN values and Scaling SVM Features ---
Shape of X_train_svm_scaled: (125973, 122)
Shape of X_test_svm_scaled: (22544, 122)

--- Training SVM Model (LinearSVC) ---

--- Evaluating SVM Model ---
SVM Accuracy: 0.8132984386089425
SVM Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.81      0.83     12833
           1       0.76      0.82      0.79      9711

    accuracy                           0.81     22544
   macro avg       0.81      0.81      0.81     22544
weighted avg       0.82      0.81      0.81     22544

✅ SVM Model metrics saved to svm_model_metrics.pkl
✅ SVM Model and Scaler saved.


In [ ]:
output of steamlit should contain follwing:
1 Data preview
2 number of noramal and attacker in graph
3 category in which data is map to (liks doS,etc) and a graph of it
4 Precision, F-1score, recall,support